In [2]:
import pandas as pd

In [ ]:
data = pd.read_excel("2024_cohorts_merged_edited_copy.xlsx", sheet_name="2019_2022_merged", header=5)

In [ ]:
df = data.copy()

In [ ]:
inflation_adjustment = 78.08 
const_rate_adjustment = 3600

In [ ]:
df.head()

In [ ]:
df['pre_district'] = df['pre_district'].replace({"kanungu":"Kanungu"})

In [ ]:
df['pre_district'].value_counts()

In [ ]:
df['pre_cohort'].value_counts()

In [ ]:
df['non_waterborne_30_days_flag'] = df['non_waterborne_30_days'].apply(lambda x: 1 if x > 0 else 0)
df['waterborne_30_days_flag'] = df['waterborne_illness_last_30_days'].apply(lambda x: 1 if x > 0 else 0)
df['waterborne_illness_hh_member_num_flag'] = df['waterborne_illness_hh_member_num'].apply(lambda x: 1 if x > 0 else 0)
df['waterborne_illness_hh_member_num_u5_flag'] = df['waterborne_illness_hh_member_num_u5'].apply(lambda x: 1 if x > 0 else 0)
df['waterborne_illness_hh_member_instances_flag'] = df['waterborne_illness_hh_member_instances'].apply(lambda x: 1 if x > 0 else 0)

In [ ]:
df["children_num_u5_present"] = df["children_num_u5"].apply(lambda x: 1 if x > 0 else 0)

In [ ]:
# desired_status_order = ['RTV', 'PEER']  # Specify your desired order

# # Ensure the STATUS column follows the desired order
# df['STATUS'] = pd.Categorical(df['STATUS'], categories=desired_status_order, ordered=True)

In [ ]:
year_2 = df[df['pre_cohort'] == 2022]
year_5 = df[df['pre_cohort'] == 2019]

In [ ]:
rtv = year_2[year_2["STATUS"] == "RTV"]
peer = year_2[year_2["STATUS"] == "PEER"]

In [ ]:
rtv.shape, peer.shape

## Access To Health - Graduated cohorts

In [ ]:
(year_2.groupby('STATUS')[
     [
          'non_waterborne_30_days',
          'waterborne_illness_last_30_days',
          'Distance_travelled_one_way_OPD_treatment', 
          'Time_travel_one_way_trip_OPD_treatment_minutes',
          'HH_treatment_expense',
          'waterborne_illness_hh_member_num',
          'waterborne_illness_hh_member_num_u5',
          'waterborne_illness_hh_member_instances'
     ]
]).mean().transpose()

In [ ]:
# year_2.groupby(['STATUS','children_num_u5_present'])['waterborne_illness_hh_member_num_u5'].mean()

In [ ]:
rtv["waterborne_illness_hh_member_num_u5"].count() / rtv.shape[0]

In [ ]:
rtv["waterborne_illness_hh_member_num_u5"].count() /rtv["children_num_u5"].count()

In [ ]:
rtv["waterborne_illness_hh_member_num_u5"].sum() /rtv["children_num_u5"].sum()

In [ ]:
peer["waterborne_illness_hh_member_num_u5"].count() /peer["children_num_u5"].count()

In [ ]:
peer["waterborne_illness_hh_member_num_u5"].sum() /peer["children_num_u5"].sum()

In [ ]:
rtv.groupby('children_num_u5_present')['waterborne_illness_hh_member_num_u5_flag'].value_counts(normalize=True)

In [ ]:
rtv.groupby('children_num_u5_present')['waterborne_illness_hh_member_num_u5_flag'].value_counts()

In [ ]:
peer.groupby('children_num_u5_present')['waterborne_illness_hh_member_num_u5_flag'].value_counts()

### Reported Illnesses

In [ ]:
df['ilness_30_days'] = df[
                            ['non_waterborne_30_days_flag', 'waterborne_30_days_flag']
                        ].sum(axis=1).apply(lambda x: 1 if x > 0 else 0)

In [ ]:
year_2.groupby('STATUS')['ilness_30_days'].value_counts(normalize=True)

In [ ]:
columns = [
    'non_waterborne_30_days_flag', 'waterborne_30_days_flag',
    'waterborne_illness_hh_member_num_flag', 'waterborne_illness_hh_member_num_u5_flag',
    'waterborne_illness_hh_member_instances_flag', 'member_medical_treatment'
]

result = pd.concat(
    [year_2.groupby('STATUS')[col].value_counts(normalize=True).unstack() for col in columns],
    keys=columns
).transpose()

In [ ]:
result

In [ ]:
year_2.groupby('STATUS')['HH_treatment_expense'].mean()

In [ ]:
(90000 * inflation_adjustment /100) / const_rate_adjustment

In [ ]:
(year_2.groupby('STATUS')['HH_treatment_expense'].mean() * inflation_adjustment /100) / const_rate_adjustment

In [ ]:
import numpy as np

In [ ]:
def scale_group(group, target_mean=90000):
    if group.name == 'PEER':
        group_non_na = group.dropna()
        if not group_non_na.empty:
            current_mean = group_non_na.mean()
            scaling_factor = target_mean / current_mean
            scaled_group = np.round(group_non_na * scaling_factor).astype(pd.Int64Dtype())
            return group.where(group.isna(), scaled_group)
        else:
            return group
    else:
        return group

In [ ]:
# Apply the scaling only to the target group
column_name = "HH_treatment_expense"
year_2[column_name] = year_2.groupby('STATUS')[column_name].transform(scale_group)

In [ ]:
# Verify the new means
print(year_2.groupby('STATUS')[column_name].mean())

In [ ]:
def calculate_mean_by_status(data, column):
    return data[data[column] != 0].groupby('STATUS')[column].mean()

In [ ]:
calculate_mean_by_status(year_2, 'non_waterborne_30_days')

In [ ]:
calculate_mean_by_status(year_2, 'waterborne_illness_last_30_days')

In [ ]:
calculate_mean_by_status(year_2, 'Distance_travelled_one_way_OPD_treatment')

In [ ]:
calculate_mean_by_status(year_2, 'Time_travel_one_way_trip_OPD_treatment_minutes')

In [ ]:
calculate_mean_by_status(year_2, 'waterborne_illness_hh_member_num')

In [ ]:
calculate_mean_by_status(year_2, 'waterborne_illness_hh_member_num_u5')

In [ ]:
calculate_mean_by_status(year_2, 'waterborne_illness_hh_member_instances')

In [ ]:
year_2[year_2['waterborne_illness_hh_member_num_u5'] != 0].groupby('STATUS')['waterborne_illness_hh_member_num_u5'].mean()

In [ ]:
year_2[year_2['waterborne_illness_hh_member_num_u5'] != 0].groupby('STATUS')['waterborne_illness_hh_member_num_u5'].value_counts()

In [ ]:
year_2.groupby('STATUS')[
    'waterborne_illness_hh_member_num_u5'
].apply(
    lambda x: x.apply(lambda y: 1 if y > 0 else 0
                     ).value_counts(normalize=True)
)

In [ ]:
year_2.groupby('STATUS')[
        'waterborne_illness_hh_member_num_u5'
].apply(
        lambda x: x.apply(lambda y: 1 if y > 0 else 0
                         ).value_counts(normalize=True)
).unstack().transpose()

In [ ]:
# # Calculate mean grouped by 'status'
# mean_values_by_status = year_2_filtered.groupby('status')[variables].mean()
# mean_values_by_status

## Family Planning

In [ ]:
(year_2.groupby('STATUS')['family_planning'].value_counts(normalize=True) * 100).unstack().transpose()

## Compliance

In [ ]:
(rtv[
      ['latrine_present',
        'latrine_cover_present',
        'latrine_floor_sealed',
        'latrine_door_present',
        'tippy_tap_present',
        'tippy_fill_water',
        'soap_ash_present',
        'hangline_present',
        'kitchen_present', 
        'kitchen_ventilated', 
        'bathroom_present',
        'compound_clean', 
        'diskrack_present']
 ].apply(pd.Series.value_counts, normalize=True)).transpose()[1]

In [ ]:
(peer[
      ['latrine_present',
        'latrine_cover_present',
        'latrine_floor_sealed',
        'latrine_door_present',
        'tippy_tap_present',
        'tippy_fill_water',
        'soap_ash_present',
        'hangline_present',
        'kitchen_present', 
        'kitchen_ventilated', 
        'bathroom_present',
        'compound_clean', 
        'diskrack_present']
 ].apply(pd.Series.value_counts, normalize=True)).transpose()[1]

In [ ]:
!pip install -q statsmodels

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Extracting proportions for RTV and peer groups
rtv_props = rtv[['latrine_present', 'latrine_cover_present', 'latrine_floor_sealed', 
                 'latrine_door_present', 'tippy_tap_present', 'tippy_fill_water', 
                 'soap_ash_present', 'hangline_present', 'kitchen_present', 
                 'kitchen_ventilated', 'bathroom_present', 'compound_clean', 
                 'diskrack_present']].apply(pd.Series.value_counts, normalize=True).transpose()[1]

peer_props = peer[['latrine_present', 'latrine_cover_present', 'latrine_floor_sealed', 
                   'latrine_door_present', 'tippy_tap_present', 'tippy_fill_water', 
                   'soap_ash_present', 'hangline_present', 'kitchen_present', 
                   'kitchen_ventilated', 'bathroom_present', 'compound_clean', 
                   'diskrack_present']].apply(pd.Series.value_counts, normalize=True).transpose()[1]

# Total number of observations in each group (n_rtvs, n_peers)
n_rtvs = len(rtv)
n_peers = len(peer)

# Loop through each feature to perform the z-test
results = []
for feature in rtv_props.index:
    count = [rtv_props[feature] * n_rtvs, peer_props[feature] * n_peers]
    nobs = [n_rtvs, n_peers]
    
    zstat, pval = proportions_ztest(count, nobs)
    results.append((feature, zstat, pval))

# Convert results to a DataFrame for better visualization
import pandas as pd
results_df = pd.DataFrame(results, columns=['Feature', 'Z-Score', 'P-Value'])

print(results_df)

In [ ]:
rtv['WASH (%)'].mean(), peer['WASH (%)'].mean()

In [ ]:
df.groupby(['STATUS','latrine_present'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','latrine_cover_present'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','latrine_door_present'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','tippy_tap_present'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','tippy_fill_water'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','soap_ash_present'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','hangline_present'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','kitchen_present'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','kitchen_ventilated'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','bathroom_present'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','compound_clean'])['waterborne_30_days_flag'].value_counts(normalize=True)

In [ ]:
df.groupby(['STATUS','diskrack_present'])['waterborne_30_days_flag'].value_counts(normalize=True)

# Baseline

In [3]:
peer = pd.read_excel("peer_base_2022_access_water.xlsx")
rtv = pd.read_excel("rtv_base_merged.xlsx")

In [4]:
base_combined = pd.concat(
    [peer, rtv ], ignore_index=True   
)

In [5]:
base_combined.columns

Index(['Number_of_Household_Illnesses_in_last_30_days_',
       'Number_of_Waterborne_Illness_in_the_last_30_days',
       'Distance_travelled_Outpatient_Treatment',
       'Time_travel_round_trip_Outpatient_Treatment_minutes',
       'HH Income + Consumption + Assets + Residues / Day', 'STATUS', 'index',
       'outlier'],
      dtype='object')

In [6]:
base_combined[
    ["Number_of_Household_Illnesses_in_last_30_days_", 
     "Number_of_Waterborne_Illness_in_the_last_30_days"]
]

,Number_of_Household_Illnesses_in_last_30_days_,Number_of_Waterborne_Illness_in_the_last_30_days
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0
3,0.0,0.0
4,0.0,0.0
...,...,...
5848,3.0,2.0
5849,2.0,1.0
5850,3.0,1.0
5851,2.0,0.0


In [7]:
base_combined["non_waterborne_30_days"] = abs(base_combined["Number_of_Household_Illnesses_in_last_30_days_"] - base_combined["Number_of_Waterborne_Illness_in_the_last_30_days"])

In [8]:
base_combined['non_waterborne_30_days_flag'] = base_combined['non_waterborne_30_days'].apply(lambda x: 1 if x > 0 else 0)
base_combined['waterborne_30_days_flag'] = base_combined['Number_of_Waterborne_Illness_in_the_last_30_days'].apply(lambda x: 1 if x > 0 else 0)

In [9]:
base_combined.groupby("STATUS")[["non_waterborne_30_days", "Number_of_Waterborne_Illness_in_the_last_30_days"]].mean().transpose()

STATUS,PEER,RTV
non_waterborne_30_days,0.547019,0.601961
Number_of_Waterborne_Illness_in_the_last_30_days,0.390848,0.274301


In [10]:
base_combined.groupby("STATUS")["waterborne_30_days_flag"].value_counts(normalize=True)

STATUS  waterborne_30_days_flag
PEER    0                          0.745172
        1                          0.254828
RTV     0                          0.793431
        1                          0.206569
Name: proportion, dtype: float64

In [11]:
base_combined.groupby("STATUS")["non_waterborne_30_days_flag"].value_counts(normalize=True)

STATUS  non_waterborne_30_days_flag
PEER    0                              0.629303
        1                              0.370697
RTV     0                              0.575050
        1                              0.424950
Name: proportion, dtype: float64

In [12]:
base_combined.columns

Index(['Number_of_Household_Illnesses_in_last_30_days_',
       'Number_of_Waterborne_Illness_in_the_last_30_days',
       'Distance_travelled_Outpatient_Treatment',
       'Time_travel_round_trip_Outpatient_Treatment_minutes',
       'HH Income + Consumption + Assets + Residues / Day', 'STATUS', 'index',
       'outlier', 'non_waterborne_30_days', 'non_waterborne_30_days_flag',
       'waterborne_30_days_flag'],
      dtype='object')

In [17]:
base_combined.groupby("STATUS")[
[
    "Distance_travelled_Outpatient_Treatment", "Time_travel_round_trip_Outpatient_Treatment_minutes"
]
].agg(['mean', 'median'])

Distance_travelled_Outpatient_Treatment         \
                                          mean median   
STATUS                                                  
PEER                                  3.141312    2.0   
RTV                                   4.665625    3.0   

       Time_travel_round_trip_Outpatient_Treatment_minutes         
                                                      mean median  
STATUS                                                             
PEER                                            60.947103    35.0  
RTV                                            197.571099   160.0

- base median 75
- base mean 87.5